In [7]:
import os
import sys
import uuid
from dotenv import load_dotenv

In [8]:
path= "/Users/shivani/kenpath/mh-oan-api"

In [9]:
os.chdir(path)

In [10]:
load_dotenv()

True

In [13]:
# Fix invalid DEBUG from shell env (e.g. DEBUG=release)
if os.getenv("DEBUG", "").lower() not in ("true", "1", "false", "0", ""):
    os.environ["DEBUG"] = "false"

In [14]:
from helpers import langfuse_helper  # noqa: F401
from langfuse import get_client, propagate_attributes
from helpers.langfuse_tracing import lf_set_trace_io, lf_update_current_observation
from helpers.langfuse_trace_schema import AGENT_VISTAAR
from app.services.chat import _run_moderation
from agents.agrinet import agrinet_agent
from agents.suggestions import suggestions_agent
from agents.deps import FarmerContext
from langcodes import Language

2026-05-27 17:01:42,631 - app.core.cache - INFO - Cache configured with Redis at redis-stack:6379 (DB: 0, Prefix: sva-cache-, Max Connections: 20, Auto-cleanup: Enabled)


In [15]:
MODEL_NAME = os.getenv("LLM_AGRINET_MODEL_NAME", "agrinet-model")
LANGFUSE_URL = os.getenv("LANGFUSE_BASE_URL", "")

In [16]:
async def answer_question(question, language="mr", farmer_id=None):
    session_id = f"notebook-{uuid.uuid4().hex[:8]}"

    deps = FarmerContext(query=question, lang_code=language, farmer_id=farmer_id)
    user_message = deps.get_user_message()
    lf = get_client()

    with propagate_attributes(session_id=session_id, trace_name="vistaar-agent"):
        with lf.start_as_current_observation(as_type="chain", name="chain.chat"):
            lf_set_trace_io(input=question)

            moderation = await _run_moderation(user_message, session_id)
            deps.update_moderation_str(str(moderation))

            user_message = deps.get_user_message()
            with lf.start_as_current_observation(as_type="agent", name=AGENT_VISTAAR):
                lf_update_current_observation(input=user_message)
                agrinet_run = await agrinet_agent.run(user_message, deps=deps)
                usage = agrinet_run.usage()
                lf_update_current_observation(
                    output=agrinet_run.output,
                    model=MODEL_NAME,
                    request_tokens=usage.input_tokens or 0,
                    response_tokens=usage.output_tokens or 0,
                )

            response = agrinet_run.output
            lf_set_trace_io(output=response)

            lang_name = Language.get(language).display_name(language)
            sugg_message = (
                f"**Conversation**\n\n"
                f"**User:** \"{question}\"\n\n"
                f"**Assistant:** {response}\n\n"
                f"**Based on the conversation, suggest 3-5 questions "
                f"the farmer can ask in {lang_name}.**"
            )
            sugg_run = await suggestions_agent.run(
                sugg_message,
                deps=FarmerContext(query=sugg_message, lang_code=language),
            )

    lf.flush()

    return {
        "session_id": session_id,
        "moderation": str(moderation),
        "response": response,
        "suggestions": sugg_run.output,
    }

In [ ]:
# question = "What is the best fertilizer for soyabean?"
# language = "mr"

In [18]:

# result = await answer_question(question, language)

# print(f"Session ID: {result['session_id']}")
# print(f"Langfuse: {LANGFUSE_URL}\n")
# print("MODERATION:", result["moderation"], sep="\n")
# print("\nRESPONSE:", result["response"], sep="\n")
# print("\nSUGGESTIONS:", result["suggestions"], sep="\n")

2026-05-27 17:01:54,006 - agents.tools.search - INFO - Searching for 'Soybean fertilizer recommendation' in index 'optimization-index'


Session ID: notebook-5c126b69
Langfuse: https://vistaar-langfuse.mahapocra.gov.in

MODERATION:
**Moderation Compliance:** ✅ Proceed with the query (Valid Agricultural)

RESPONSE:
सोयाबीनसाठी खत पेरणीच्या वेळी पूर्ण देणे आणि शेंग बनण्याच्या टप्प्यावर योग्य फवारणी करणे हे सर्वोत्तम राखले आहे.

**खत व्यवस्थापन:** पेरणीच्या वेळी प्रति हेक्टर ३० कि.ग्रा नत्र, ६० कि.ग्रा स्फुरद व ३० कि.ग्रा पालाश द्या (म्हणजे ६५ कि.ग्रा यूरिया, ३७५ कि.ग्रा सिंगल सुपर फॉस्फेट, ५० कि.ग्रा म्युरिएट ऑफ पोटॅश).  
**जैविक खत/प्रक्रिया:** बियाणे जीवाणू खताने प्रक्रिया केल्यास रासायनिक खताची अर्धी मात्रा बचत होते व मातीची सुपीकता टिकते; शक्य असल्यास पेरणीपूर्वी ५ टन कुजलेले शेणखत किंवा कंपोस्ट प्रति हेक्टर वापरा.  
**गंधक:** मिश्रित खते वापरत असाल तर पेरणीवेळी २० कि.ग्रा गंधक प्रति हेक्टर द्या.  
**शेंगांच्या अवस्थेत फवारणी:** शेंग बनण्याच्या टप्प्यावर शिफारसीत डोस खते आणि २% डी.ए.पी. किंवा २% १९:१९:१९ यापैकी कोणतेही एक पाण्यात मिसळून फवारणी करा.  
**माती तपासणी:** मात्रा ठरवताना माती चाचणी अहवाल पाहा आणि आवश्यक असल

In [21]:
question = "सोयाबीन पिकाची पाने पिवळी पडत असल्यास उपाय काय करावा?"
language = "mr"

for i in range(1, 20):
    result = await answer_question(question, language)
    print(f"Run {i}/100 done — {result['session_id']}")

2026-05-27 17:07:55,633 - agents.tools.search - INFO - Searching for 'Soybean leaf chlorosis management' in index 'optimization-index'
2026-05-27 17:07:57,888 - agents.tools.search - INFO - Searching for 'Soybean yellow mosaic whitefly management' in index 'optimization-index'
2026-05-27 17:08:00,654 - agents.tools.search - INFO - Searching for 'Soybean yellow mosaic disease management Punjabrao Deshmukh Krishi Vidyapeeth' in index 'optimization-index'
2026-05-27 17:08:03,396 - agents.tools.search - INFO - Searching for 'Whitefly soybean insecticide Acetamiprid Bifenthrin Punjabrao Deshmukh Krishi Vidyapeeth' in index 'optimization-index'


Run 1/100 done — notebook-102cd8e4


2026-05-27 17:08:17,674 - agents.tools.search - INFO - Searching for 'Soybean leaf chlorosis yellowing management' in index 'optimization-index'
2026-05-27 17:08:19,033 - agents.tools.search - INFO - Searching for 'Soybean yellow mosaic virus whitefly management' in index 'optimization-index'


Run 2/100 done — notebook-4965233d


2026-05-27 17:08:29,208 - agents.tools.search - INFO - Searching for 'Soybean leaf chlorosis management' in index 'optimization-index'


...

Run 17/100 done — notebook-d9393795


2026-05-27 17:12:00,589 - agents.tools.search - INFO - Searching for 'Soybean Yellowing Leaves Nutrient Deficiency Disease Management' in index 'optimization-index'
2026-05-27 17:12:03,188 - agents.tools.search - INFO - Searching for 'Soybean Yellow Mosaic Disease Leaf Yellowing Management' in index 'optimization-index'


Run 18/100 done — notebook-707f5a7f


2026-05-27 17:12:17,540 - agents.tools.search - INFO - Searching for 'Soybean leaf chlorosis yellowing nutrient deficiency management' in index 'optimization-index'


Run 19/100 done — notebook-ba5aee09


In [ ]:
# questions = [
#     "यावर्षी पावसाचे प्रमाण कमी आहे तर उताराला कशी पेरणी करावी",
#     "मराठवाड्यासाठी तूर पिकाच्या कोणत्या सुधारित जाती शिफारसीय आहेत?",
#     "सोयाबीन पिकाची पाने पिवळी पडत असल्यास उपाय काय करावा?",
#     "गुलाबी बोंडअळी नियंत्रणासाठी कापसावर कोणती फवारणी करावी?",
#     "गोदामांची स्वच्छता करण्यासाठी कोणते फ्युमिगंट वापरावे?",
#     "पुढील ५ दिवसांत पुणे जिल्ह्यात पावसाचा अंदाज असल्यास फवारणी करावी का?",
#     "उष्णतेच्या लाटेमुळे फळबागांचे संरक्षण कसे करावे?",
#     "हरभऱ्यातील घटाळे चे नियंत्रण कसे करावे",
#     "हळद व आले पिकाची लागवड कधी व कशी करावी",
#     "कमी दिवसात पीक तयार होणाऱ्या वाणांची माहिती कुठे मिळेल",
# ]